# Week 12, Day 3: Project: a gradebook

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w012d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('students.csv', 'w', encoding='utf-8') as f:
    f.write('id,name,class\nS01,Ada Obi,3A\nS02,Bo Musa,3A\nS03,Cy Bello,3B\nS04,Di Eze,3B\nS05,Ed Okoro,3A\nS06,Fe Ade,3B\n')
with open('scores.csv', 'w', encoding='utf-8') as f:
    f.write('student,subject,score\nS01,maths,93\nS01,english,71\nS01,science,88\nS02,maths,41\nS02,english,88\nS02,science,57\nS03,maths,78\nS03,english,64\nS03,science,\nS04,maths,55\nS04,english,90\nS04,science,72\nS05,maths,67\nS05,english,abc\nS05,science,49\nS07,maths,80\nS02,maths,45\nS06,maths,101\nS06,english,58\nS06,science,61\nS06,art,70\n')

*Interactive figure on the website: The gradebook's plan. Each box is a function; arrows show which function calls which. Yellow boxes are stubs, not written yet..*

> **Key idea.** Plan the data and the functions before the code. Write each function on its own, check it on small examples, and only then join them up. When something goes wrong, you'll know which piece to look at.

### Exercise: Step 1: load the students

Write `load_students(path)` that reads a students file with `csv.DictReader` and returns a dictionary from each `id` to a dictionary with the student's `name` and `class`. Strip spaces from every value.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_file():\n    \"students.csv\"\n    s = load_students(\"students.csv\")\n    assert s[\"S01\"] == {\"name\": \"Ada Obi\", \"class\": \"3A\"}\n    assert list(s) == [\"S01\", \"S02\", \"S03\", \"S04\", \"S05\", \"S06\"]\ndef test_spaces():\n    \"spaces are stripped\"\n    write_file(\"s2.csv\", \"id,name,class\\n S9 , Zed Ola ,3C \\n\")\n    assert load_students(\"s2.csv\") == {\"S9\": {\"name\": \"Zed Ola\", \"class\": \"3C\"}}")

### Exercise: Step 2: check one score

Write `check_score(text)` that takes the score field as text and returns it as an int, raising a `ValueError` with one of these messages otherwise:

- `missing score` if it's empty or only spaces;
- `not a whole number: '12.5'` (the stripped text, with quotes, as `{text!r}` gives) if it isn't all digits;
- `out of range: 101` if it's over 100.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def message(text):\n    try:\n        check_score(text)\n    except ValueError as e:\n        return str(e)\n    raise AssertionError(f\"check_score({text!r}) should raise a ValueError\")\ndef test_good():\n    \"good scores\"\n    assert check_score(\"93\") == 93 and check_score(\" 0 \") == 0 and check_score(\"100\") == 100\ndef test_bad():\n    \"bad scores\"\n    assert message(\"\") == \"missing score\"\n    assert message(\"   \") == \"missing score\"\n    assert message(\"abc\") == \"not a whole number: 'abc'\"\n    assert message(\" 12.5 \") == \"not a whole number: '12.5'\"\n    assert message(\"-4\") == \"not a whole number: '-4'\"\n    assert message(\"101\") == \"out of range: 101\"")

### Exercise: Step 3: load the scores

Write `load_scores(path, students)` that returns a tuple `(scores, problems)`:

- `scores` is a dictionary from **every** student id in `students` to a dictionary from subject to score (empty if they have none);
- `problems` is a list of messages like `line 10: missing score`, where the line number counts the header as line 1.

For each row, strip the student id, and strip and lower-case the subject, then check in this order:

1. the student isn't in `students`: `unknown student 'S07'`;
2. the subject isn't in `SUBJECTS`: `unknown subject 'art'`;
3. the student already has a score for that subject: `duplicate maths score for S02` (the first one is kept);
4. the score itself, with `check_score`.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def load_students(path):
    students = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            students[row["id"].strip()] = {"name": row["name"].strip(), "class": row["class"].strip()}
    return students

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scores():\n    \"scores\"\n    students = load_students(\"students.csv\")\n    scores, problems = load_scores(\"scores.csv\", students)\n    assert scores[\"S01\"] == {\"maths\": 93, \"english\": 71, \"science\": 88}\n    assert scores[\"S02\"][\"maths\"] == 41, \"the first of two duplicate marks should be kept\"\n    assert scores[\"S06\"] == {\"english\": 58, \"science\": 61}\n    assert list(scores) == [\"S01\", \"S02\", \"S03\", \"S04\", \"S05\", \"S06\"], \"every student needs an entry, even with no scores\"\ndef test_problems():\n    \"problems\"\n    students = load_students(\"students.csv\")\n    scores, problems = load_scores(\"scores.csv\", students)\n    assert problems == [\"line 10: missing score\", \"line 15: not a whole number: 'abc'\", \"line 17: unknown student 'S07'\",\n                        \"line 18: duplicate maths score for S02\", \"line 19: out of range: 101\", \"line 22: unknown subject 'art'\"], \"Got:\\n\" + \"\\n\".join(problems)\ndef test_empty():\n    \"a student with no scores\"\n    write_file(\"sc.csv\", \"student,subject,score\\n\")\n    scores, problems = load_scores(\"sc.csv\", {\"S1\": {\"name\": \"A\", \"class\": \"1\"}})\n    assert scores == {\"S1\": {}} and problems == []")

> **Why raise inside the loop?.** The three checks in `load_scores` raise their own `ValueError`s, and `check_score` raises its own. All four arrive at the same `except`, which turns them into a problem message. One handler, one format for every message, and adding a fifth check later is one more `if` and `raise`.

### Exercise: Step 4: average and grade

Write `average(subject_scores)`, which takes one student's dictionary of scores and returns their mean rounded to 1 place, or `None` if it's empty; and `grade(avg)`, which returns the letter for an average using `BOUNDARIES`, or `"-"` for `None`.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_average():\n    \"average\"\n    assert average({\"maths\": 93, \"english\": 71, \"science\": 88}) == 84.0\n    assert average({\"maths\": 55, \"english\": 90, \"science\": 72}) == 72.3\n    assert average({}) is None\ndef test_grade():\n    \"grade\"\n    assert [grade(a) for a in (84.0, 70, 69.9, 60, 50, 49.9, 45, 40, 39.9, 0)] == [\"A\", \"A\", \"B\", \"B\", \"C\", \"D\", \"D\", \"E\", \"F\", \"F\"]\n    assert grade(None) == \"-\"")

### Exercise: Step 5: summarise

Write `summarise(students, scores)` that returns a list with one dictionary per student, with the keys `id`, `name`, `class`, `average`, `grade` and `missing` (the list of subjects in `SUBJECTS` order that the student has no score for), sorted by class and then by name.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def load_students(path):
    students = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            students[row["id"].strip()] = {"name": row["name"].strip(), "class": row["class"].strip()}
    return students

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

def load_scores(path, students):
    scores = {sid: {} for sid in students}
    problems = []
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for line, row in enumerate(reader, start=2):
            sid, subject = row["student"].strip(), row["subject"].strip().lower()
            try:
                if sid not in students:
                    raise ValueError(f"unknown student {sid!r}")
                if subject not in SUBJECTS:
                    raise ValueError(f"unknown subject {subject!r}")
                if subject in scores[sid]:
                    raise ValueError(f"duplicate {subject} score for {sid}")
                scores[sid][subject] = check_score(row["score"])
            except ValueError as e:
                problems.append(f"line {line}: {e}")
    return scores, problems

def average(subject_scores):
    if not subject_scores:
        return None
    return round(sum(subject_scores.values()) / len(subject_scores), 1)

def grade(avg):
    if avg is None:
        return "-"
    for lowest, letter in BOUNDARIES:
        if avg >= lowest:
            return letter
    return "F"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_summary():\n    \"the summary\"\n    students = load_students(\"students.csv\")\n    scores, problems = load_scores(\"scores.csv\", students)\n    rows = summarise(students, scores)\n    assert [r[\"name\"] for r in rows] == [\"Ada Obi\", \"Bo Musa\", \"Ed Okoro\", \"Cy Bello\", \"Di Eze\", \"Fe Ade\"]\n    assert rows[2] == {\"id\": \"S05\", \"name\": \"Ed Okoro\", \"class\": \"3A\", \"average\": 58.0, \"grade\": \"C\", \"missing\": [\"english\"]}\n    assert rows[5][\"missing\"] == [\"maths\"] and rows[5][\"average\"] == 59.5\ndef test_nobody():\n    \"a student with no marks\"\n    rows = summarise({\"S9\": {\"name\": \"Zed\", \"class\": \"1\"}}, {\"S9\": {}})\n    assert rows == [{\"id\": \"S9\", \"name\": \"Zed\", \"class\": \"1\", \"average\": None, \"grade\": \"-\", \"missing\": [\"maths\", \"english\", \"science\"]}]")

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def load_students(path):
    students = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            students[row["id"].strip()] = {"name": row["name"].strip(), "class": row["class"].strip()}
    return students

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

def load_scores(path, students):
    scores = {sid: {} for sid in students}
    problems = []
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for line, row in enumerate(reader, start=2):
            sid, subject = row["student"].strip(), row["subject"].strip().lower()
            try:
                if sid not in students:
                    raise ValueError(f"unknown student {sid!r}")
                if subject not in SUBJECTS:
                    raise ValueError(f"unknown subject {subject!r}")
                if subject in scores[sid]:
                    raise ValueError(f"duplicate {subject} score for {sid}")
                scores[sid][subject] = check_score(row["score"])
            except ValueError as e:
                problems.append(f"line {line}: {e}")
    return scores, problems

def average(subject_scores):
    if not subject_scores:
        return None
    return round(sum(subject_scores.values()) / len(subject_scores), 1)

def grade(avg):
    if avg is None:
        return "-"
    for lowest, letter in BOUNDARIES:
        if avg >= lowest:
            return letter
    return "F"

def summarise(students, scores):
    rows = []
    for sid, info in students.items():
        avg = average(scores[sid])
        rows.append({"id": sid, "name": info["name"], "class": info["class"], "average": avg,
                     "grade": grade(avg), "missing": [s for s in SUBJECTS if s not in scores[sid]]})
    return sorted(rows, key=lambda r: (r["class"], r["name"]))

students = load_students("students.csv")
scores, problems = load_scores("scores.csv", students)
print(f"{len(problems)} problem(s):")
for p in problems:
    print(" ", p)
print()
for r in summarise(students, scores):
    avg = "-" if r["average"] is None else r["average"]
    note = f"  (no {', '.join(r['missing'])})" if r["missing"] else ""
    print(f"{r['class']}  {r['name']:<10}{avg:>6}  {r['grade']}{note}")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## The brief

A school wants a small program to process each term's marks. Two files come from its records system every term:

- `students.csv`: one row per student, with an `id`, a `name` and a `class`;
- `scores.csv`: one row per mark, with the `student` id, the `subject` (`maths`, `english` or `science`) and the `score`, a whole number from 0 to 100.

The records system isn't perfect. Marks are sometimes missing, mistyped, out of range, entered twice, entered for students who don't exist, or entered for subjects the school doesn't teach. The program must **never crash** on bad data, and must **never silently drop** a mark: every problem is reported with its line number, and everything else is used.

For each student, the program works out the average of the marks they have (rounded to 1 place) and a grade from that average, using the boundaries from Week 12, Day 1: A from 70, B from 60, C from 50, D from 45, E from 40, otherwise F. A student with no valid marks has no average and the grade `-`. Tomorrow you'll add class reports, report cards saved as files, tests, and a menu.

Open both files and look through `scores.csv` now. Before reading on, try to spot every problem in it. (There are six.)

## A plan

Don't start by writing code. Start by deciding **what pieces** the program needs, what each one takes and gives back, and how they fit together. This is **top-down design** from Week 7: the top level reads like the brief, and each function does one job.

The data first. What shape suits each part?

- **Students**: we'll look students up by id all the time, so a dictionary from id to that student's details: `{"S01": {"name": "Ada Obi", "class": "3A"}, ...}`.
- **Scores**: for each student, a mark per subject, so a dictionary of dictionaries: `{"S01": {"maths": 93, "english": 71, ...}, ...}`.
- **Problems**: a list of messages, in file order.

Then the functions. Here's the plan as a structure chart. The yellow boxes are stubs, the functions you'll write today:

Each step below is one function. Each step's starting code includes **model versions** of the functions from earlier steps, so you can do the steps in any order, and a mistake in one step doesn't block the next.

## Step 1: students

## Step 2: checking a score

Checking a single score is a job of its own, so it gets its own function. It follows the advice from Day 5 of last week: return the good value, or **raise** a `ValueError` saying exactly what's wrong, and let the caller decide what to do about it.

## Step 3: scores and problems

Now the heart of the loading: read every row of `scores.csv`, check it, and either store the score or record the problem. The order of the checks matters, because each row gets **one** message, for the first problem found.

## Step 4: averages and grades

## Step 5: the summary

## Checkpoint

Put it together. Here is the whole first version, using the model functions. Run it, and compare what it prints with the problems you spotted in `scores.csv` at the start.

It already does the job the school asked for, and every number in it came from a function you've checked on its own.

## Quiz

## Before you go

Look at your own Step 3. Could someone else read it and see each check clearly? If a function is longer than about 20 lines, see whether part of it could be its own function. Tomorrow: class reports, report cards saved to files, tests, and a menu to drive it all.